Before we start, we need to make sure that we have a Kafka cluster running and a topic that produces some streaming data. For simplicity, we will use a single-node Kafka cluster and a topic named `users`. Open the `4.0 user-gen-kafka.ipynb` notebook and execute the cell. This notebook produces a user record every few seconds and put it on a Kafka topic called users. 

In [1]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

In [2]:
builder = (SparkSession.builder
           .appName("config-checkpoints")
           .master("spark://spark-master:7077")
           .config("spark.executor.memory", "512m")
           .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
           .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))

spark = configure_spark_with_delta_pip(builder,['org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.1']).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-62b17beb-3b8b-413f-87e5-8c2a5daa58fd;1.0
	confs: [default]
	found io.delta#delta-core_2.12;2.4.0 in central
	found io.delta#delta-storage;2.4.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.4.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.4.1 in central
	found org.apache.kafka#kafka-clients;3.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.1 in central
	found org.slf4j#slf4j-api;2.0.6 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in centra

In [3]:
get_ipython().run_line_magic('load_ext', 'sparksql_magic')
get_ipython().run_line_magic('config', 'SparkSql.limit=20')

In [4]:
df = (spark.readStream
      .format("kafka")
      .option("kafka.bootstrap.servers", "kafka:9092")
      .option("subscribe", "users")
      .option("startingOffsets", "earliest")
      .load())

In [5]:
schema = StructType([
    StructField('id', IntegerType(), True),
    StructField('name', StringType(), True),
    StructField('age', IntegerType(), True),
    StructField('gender', StringType(), True),
    StructField('country', StringType(), True)])

df = df.withColumn('value', from_json(col('value').cast("STRING"), schema))

In [6]:
df = df.select(
    col('value.id').alias('id'),
    col('value.name').alias('name'),
    col('value.age').alias('age'),
    col('value.gender').alias('gender'),
    col('value.country').alias('country'))

In [7]:
query = (df.writeStream
   .format("console")
   .outputMode("append")
   .option("checkpointLocation", "/opt/workspace/data/checkpoint")
   .start())

-------------------------------------------
Batch: 0
-------------------------------------------
+---+------+---+------+---------+
| id|  name|age|gender|  country|
+---+------+---+------+---------+
| 18|user67| 56|     F|   Brazil|
| 95|user35| 38|     F|      USA|
| 98|user80| 58|     M|   Brazil|
| 59|user78| 61|     F|Australia|
| 51|user22| 31|     F|Australia|
| 40|user60| 39|     M|    China|
| 30| user3| 53|     F|       UK|
| 37|user69| 59|     M|Australia|
| 79|user56| 21|     F|       UK|
| 23|user25| 25|     M|    China|
| 62|user56| 60|     M|    India|
| 35|user18| 52|     F|    India|
| 64|user47| 62|     F|    China|
| 20| user2| 24|     F|      USA|
| 62|user66| 55|     F|   Canada|
| 75|user43| 19|     M|   Canada|
| 32|user73| 37|     F|    China|
| 36|user22| 28|     F|   Brazil|
| 13|user20| 41|     M|Australia|
| 52|user30| 38|     M|    India|
+---+------+---+------+---------+
only showing top 20 rows



-------------------------------------------
Batch: 1
-------------------------------------------
+---+-------+---+------+-------+
| id|   name|age|gender|country|
+---+-------+---+------+-------+
| 80|user100| 58|     F|     UK|
+---+-------+---+------+-------+

-------------------------------------------
Batch: 2
-------------------------------------------
+---+------+---+------+-------+
| id|  name|age|gender|country|
+---+------+---+------+-------+
| 32|user66| 52|     M|    USA|
+---+------+---+------+-------+



-------------------------------------------
Batch: 3
-------------------------------------------
+---+------+---+------+-------+
| id|  name|age|gender|country|
+---+------+---+------+-------+
| 65|user13| 62|     F| Brazil|
+---+------+---+------+-------+



-------------------------------------------
Batch: 4
-------------------------------------------
+---+------+---+------+-------+
| id|  name|age|gender|country|
+---+------+---+------+-------+
| 19|user55| 56|     M|     UK|
+---+------+---+------+-------+

-------------------------------------------
Batch: 5
-------------------------------------------
+---+------+---+------+-------+
| id|  name|age|gender|country|
+---+------+---+------+-------+
| 58|user76| 65|     F|  India|
+---+------+---+------+-------+

-------------------------------------------
Batch: 6
-------------------------------------------
+---+-----+---+------+-------+
| id| name|age|gender|country|
+---+-----+---+------+-------+
| 38|user2| 31|     M|  India|
+---+-----+---+------+-------+



In [8]:
query.stop()

In [9]:
query = (df.writeStream 
   .format("console") 
   .outputMode("append") 
   .option("checkpointLocation", "/opt/workspace/data/checkpoint") 
   .start()) 

-------------------------------------------
Batch: 7
-------------------------------------------
+---+------+---+------+-------+
| id|  name|age|gender|country|
+---+------+---+------+-------+
| 59|user78| 37|     M|  China|
| 76|user23| 32|     F|    USA|
| 99|user66| 47|     M|     UK|
+---+------+---+------+-------+

-------------------------------------------
Batch: 8
-------------------------------------------
+---+------+---+------+-------+
| id|  name|age|gender|country|
+---+------+---+------+-------+
| 95|user16| 49|     M|    USA|
+---+------+---+------+-------+

-------------------------------------------
Batch: 9
-------------------------------------------
+---+------+---+------+-------+
| id|  name|age|gender|country|
+---+------+---+------+-------+
| 99|user44| 41|     F| Canada|
+---+------+---+------+-------+

-------------------------------------------
Batch: 10
-------------------------------------------
+---+------+---+------+-------+
| id|  name|age|gender|country|


-------------------------------------------
Batch: 11
-------------------------------------------
+---+------+---+------+-------+
| id|  name|age|gender|country|
+---+------+---+------+-------+
| 71|user75| 18|     F|  India|
+---+------+---+------+-------+

-------------------------------------------
Batch: 12
-------------------------------------------
+---+------+---+------+-------+
| id|  name|age|gender|country|
+---+------+---+------+-------+
| 61|user82| 33|     M|  India|
+---+------+---+------+-------+

-------------------------------------------
Batch: 13
-------------------------------------------
+---+------+---+------+---------+
| id|  name|age|gender|  country|
+---+------+---+------+---------+
| 17|user51| 65|     M|Australia|
+---+------+---+------+---------+

-------------------------------------------
Batch: 14
-------------------------------------------
+---+------+---+------+-------+
| id|  name|age|gender|country|
+---+------+---+------+-------+
| 80|user20| 34|   

-------------------------------------------
Batch: 15
-------------------------------------------
+---+------+---+------+---------+
| id|  name|age|gender|  country|
+---+------+---+------+---------+
|  4|user61| 22|     F|Australia|
+---+------+---+------+---------+

-------------------------------------------
Batch: 16
-------------------------------------------
+---+------+---+------+-------+
| id|  name|age|gender|country|
+---+------+---+------+-------+
| 92|user43| 24|     M| Canada|
+---+------+---+------+-------+

-------------------------------------------
Batch: 17
-------------------------------------------
+---+------+---+------+-------+
| id|  name|age|gender|country|
+---+------+---+------+-------+
| 52|user32| 64|     M|  India|
+---+------+---+------+-------+

-------------------------------------------
Batch: 18
-------------------------------------------
+---+------+---+------+-------+
| id|  name|age|gender|country|
+---+------+---+------+-------+
| 63|user93| 42|   

-------------------------------------------
Batch: 20
-------------------------------------------
+---+------+---+------+-------+
| id|  name|age|gender|country|
+---+------+---+------+-------+
| 78|user32| 20|     M|  China|
+---+------+---+------+-------+



-------------------------------------------
Batch: 21
-------------------------------------------
+---+------+---+------+-------+
| id|  name|age|gender|country|
+---+------+---+------+-------+
| 62|user34| 25|     M|     UK|
+---+------+---+------+-------+

-------------------------------------------
Batch: 22
-------------------------------------------
+---+------+---+------+-------+
| id|  name|age|gender|country|
+---+------+---+------+-------+
| 49|user96| 30|     M|     UK|
+---+------+---+------+-------+

-------------------------------------------
Batch: 23
-------------------------------------------
+---+-----+---+------+---------+
| id| name|age|gender|  country|
+---+-----+---+------+---------+
| 58|user9| 33|     F|Australia|
+---+-----+---+------+---------+

-------------------------------------------
Batch: 24
-------------------------------------------
+---+------+---+------+---------+
| id|  name|age|gender|  country|
+---+------+---+------+---------+
|  8|user62| 45|  

-------------------------------------------
Batch: 25
-------------------------------------------
+---+------+---+------+-------+
| id|  name|age|gender|country|
+---+------+---+------+-------+
|  8|user32| 44|     F|  China|
+---+------+---+------+-------+

-------------------------------------------
Batch: 26
-------------------------------------------
+---+------+---+------+-------+
| id|  name|age|gender|country|
+---+------+---+------+-------+
|  9|user82| 18|     M| Canada|
+---+------+---+------+-------+

-------------------------------------------
Batch: 27
-------------------------------------------
+---+------+---+------+-------+
| id|  name|age|gender|country|
+---+------+---+------+-------+
| 49|user61| 39|     F|  China|
+---+------+---+------+-------+



In [11]:
query.stop()

24/02/04 18:00:17 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 14, writer: ConsoleWriter[numRows=20, truncate=true]] is aborting.
24/02/04 18:00:17 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 14, writer: ConsoleWriter[numRows=20, truncate=true]] aborted.


In [12]:
spark.stop() 